In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import copy
import math

import pandas as pd

# ------------------------
# PARAMETERS
# ------------------------
WIDTH = 380
HEIGHT = 280
NUM_VOTERS = 1000       # Can change
NUM_PARTIES = 3       # Can change
NUM_DISTRICTS = 9
VOTERS_PER_DISTRICT = NUM_VOTERS // NUM_DISTRICTS
NUM_RUNS = 10           # Number of redistricting simulations

NUM_CITIES = 8
CITY_INTENSITY = (5, 25)
CITY_SPREAD = (20, 60)

np.random.seed(1)

PARTY_IDS = [f"Party {i}" for i in range(NUM_PARTIES)]
party_colors = np.random.rand(NUM_PARTIES, 3)
PARTY_COLORS = dict(zip(PARTY_IDS, party_colors))

#-------------------------
# PARTY DISTRIBUTION (bias) - Must sum to 1 - MUST CHANGE WHEN CHANGING NUMBER OF PARTIES
# ------------------------
PARTY_BIAS = [0.4, 0.3, 0.3]  

# ------------------------
# GENERATE POPULATION DENSITY
# ------------------------
density = np.full((HEIGHT, WIDTH), 1.0)
X, Y = np.meshgrid(np.arange(WIDTH), np.arange(HEIGHT))

for _ in range(NUM_CITIES):
    cx = np.random.uniform(0, WIDTH)
    cy = np.random.uniform(0, HEIGHT)
    intensity = np.random.uniform(*CITY_INTENSITY)
    sigma = np.random.uniform(*CITY_SPREAD)
    density += intensity * np.exp(-((X - cx)**2 + (Y - cy)**2) / (2 * sigma**2))

# ------------------------
# GENERATE VOTERS BASED ON DENSITY
# ------------------------
flat_density = density.ravel()
flat_density /= flat_density.sum()
indices = np.random.choice(WIDTH * HEIGHT, size=NUM_VOTERS, p=flat_density)
voters_y, voters_x = np.unravel_index(indices, (HEIGHT, WIDTH))
voters = np.column_stack((voters_x, voters_y))


# ------------------------
# FIXED PARTY PREFERENCES
# ------------------------

voter_parties = np.random.choice(
    np.arange(NUM_PARTIES),
    size=NUM_VOTERS,
    p=PARTY_BIAS
)

#voter_parties = np.random.randint(0, NUM_PARTIES, NUM_VOTERS)

voter_colors = party_colors[voter_parties]

def connected_districts(voters, num_districts, voters_per_district, rng, k_neighbors=20):
#def connected_districts(voters, num_districts, voters_per_district, k_neighbors=20):
    N = len(voters)
    district = -np.ones(N, dtype=int)
    tree = cKDTree(voters)
    unassigned = set(range(N))

    # Create districts with BFS
    for d in range(num_districts):
        if not unassigned:
            break

        #seed = np.random.choice(list(unassigned))
        seed = rng.choice(list(unassigned))
        district[seed] = d
        unassigned.remove(seed)
        queue = [seed]
        count = 1

        while queue and count < voters_per_district:
            current = queue.pop(0)
            _, neighbors = tree.query(voters[current], k=k_neighbors)

            for n in neighbors:
                if n in unassigned:
                    district[n] = d
                    unassigned.remove(n)
                    queue.append(n)
                    count += 1
                    if count >= voters_per_district:
                        break

        # Retry if district incomplete
        if count < voters_per_district:
            for i in range(N):
                if district[i] == d:
                    district[i] = -1
                    unassigned.add(i)
            #return connected_districts(voters, num_districts, voters_per_district)
            return connected_districts(voters, num_districts, voters_per_district, rng, k_neighbors)
    # ------------------------
    # Assign leftover voters to nearest district
    # ------------------------
    leftover_indices = np.where(district == -1)[0]
    assigned_indices = np.where(district != -1)[0]

    if len(leftover_indices) > 0:
        assigned_tree = cKDTree(voters[assigned_indices])
        for i in leftover_indices:
            _, idx = assigned_tree.query(voters[i])
            nearest_assigned = assigned_indices[idx]
            district[i] = district[nearest_assigned]

    return district
# ------------------------
# COUNT VOTES
# ------------------------
def count_votes(district_labels):
    district_counts = []
    for d in range(NUM_DISTRICTS):
        mask = district_labels == d
        counts = {party: 0 for party in PARTY_IDS}
        for i in np.where(mask)[0]:
            counts[PARTY_IDS[voter_parties[i]]] += 1
        district_counts.append(counts)
    return district_counts

def efficiency_gap(district_counts):
    """
    Returns a list of efficiency gap values,
    one per district (for a single run).
    """

    district_gaps = []

    for district in district_counts:

        # Winning party
        winning_party = max(district.items(), key=lambda item: item[1])[0]
        num_winning_votes = district[winning_party]

        # Total votes in district
        total_votes = sum(district.values())

        # Losing votes
        num_losing_votes = total_votes - num_winning_votes

        # Wasted votes (multi-party generalization)
        wasted_winner = num_winning_votes - (total_votes / NUM_PARTIES)
        wasted_loser = num_losing_votes

        gap = (wasted_winner - wasted_loser) / total_votes

        district_gaps.append(gap)

    return district_gaps

'''# ------------------------
# EFFICIENCY GAP (2-party only)
# ------------------------
def efficiency_gap(district_counts):
    if NUM_PARTIES != 2:
        return None
    wasted = {party: 0 for party in PARTY_IDS}
    total_votes = 0

    for d in district_counts:
        total = sum(d.values())
        total_votes += total
        winner = max(d, key=d.get)
        for party in PARTY_IDS:
            if party == winner:
                wasted[party] += d[party] - total / 2
            else:
                wasted[party] += d[party]

    return (wasted[PARTY_IDS[0]] - wasted[PARTY_IDS[1]]) / total_votes
'''
def get_is_winner(district_counts, target_party):
    """
    Returns True if target_party has more votes than all others.
    If there is any tie for the maximum, the target party loses.
    """
    target_votes = district_counts[target_party]
    other_votes = [district_counts[p] for p in district_counts if p != target_party]
    
    # Target must be strictly greater than every other party
    return all(target_votes > v for v in other_votes)

def simulate_vote_shifts_forward(district_counts, target_party, PARTY_IDS, NUM_DISTRICTS):
    counts = [d.copy() for d in district_counts]
    competitors = [p for p in PARTY_IDS if p != target_party]
    num_competitors = len(competitors)
    
    def count_wins():
        return sum(1 for d in counts if get_is_winner(d, target_party))

    current_wins = count_wins()
    history = []

    while True:
        any_shift = False
        for d in counts:
            # 1. Identify parties with votes, sorted by lowest count
            # Use party ID as a secondary sort key to handle ties consistently
            active_competitors = sorted(
                [p for p in competitors if d[p] > 0],
                key=lambda p: (d[p], p) 
            )
            
            if active_competitors:
                # 2. Take up to (num_competitors) total votes from the lowest party first
                to_take = num_competitors
                from_party = active_competitors[0]
                
                # If the lowest doesn't have enough, we take what they have and stop 
                # (or we could cascade, but "taking from the same one" implies a single source per step)
                actual_taken = min(d[from_party], to_take)
                
                if actual_taken > 0:
                    d[from_party] -= actual_taken
                    d[target_party] += actual_taken
                    any_shift = True
        
        new_wins = count_wins()
        if new_wins > current_wins:
            vote_share = sum(d[target_party] for d in counts) / sum(sum(d.values()) for d in counts)
            history.append({"districts_won": new_wins, "vote_share": vote_share})
            current_wins = new_wins
            
        if not any_shift or new_wins == NUM_DISTRICTS:
            break
            
    return {"history": history, "initial_wins": count_wins(), "initial_vote_share": 0}

def simulate_vote_shifts_backward(district_counts, target_party, PARTY_IDS):
    counts = [d.copy() for d in district_counts]
    competitors = [p for p in PARTY_IDS if p != target_party]
    num_competitors = len(competitors)
    
    def count_wins():
        return sum(1 for d in counts if get_is_winner(d, target_party))

    current_wins = count_wins()
    history = []

    while True:
        any_shift = False
        for d in counts:
            # 1. Identify the 'Second Place' party (the strongest competitor)
            # Sort competitors by votes descending
            ranked_competitors = sorted(
                competitors,
                key=lambda p: (d[p], p),
                reverse=True
            )
            strongest_competitor = ranked_competitors[0]
            
            # 2. Move votes from target to that specific strongest competitor
            to_move = min(d[target_party], num_competitors)
            
            if to_move > 0:
                d[target_party] -= to_move
                d[strongest_competitor] += to_move
                any_shift = True
        
        new_wins = count_wins()
        if new_wins < current_wins:
            vote_share = sum(d[target_party] for d in counts) / sum(sum(d.values()) for d in counts)
            history.append({"districts_won": new_wins, "vote_share": vote_share})
            current_wins = new_wins
            
        if not any_shift or new_wins == 0:
            break
            
    return {"history": history, "initial_wins": count_wins(), "initial_vote_share": 0}

# 1. Update your simulation functions to return the current vote share
def get_current_vote_share(district_counts, target_party):
    total = sum(sum(d.values()) for d in district_counts)
    party_total = sum(d[target_party] for d in district_counts)
    return party_total / total

# 2. Updated compute function
def compute_step_curve_for_party(district_counts, target_party, PARTY_IDS, NUM_DISTRICTS):
    # Calculate starting state
    initial_vote = get_current_vote_share(district_counts, target_party)
    initial_wins = sum(1 for d in district_counts if get_is_winner(d, target_party))
    
    fwd = simulate_vote_shifts_forward(district_counts, target_party, PARTY_IDS, NUM_DISTRICTS)
    rev = simulate_vote_shifts_backward(district_counts, target_party, PARTY_IDS)

    # Base point is the current status of the party
    base_point = (initial_vote, initial_wins)

    fwd_points = [base_point] + [(h["vote_share"], h["districts_won"]) for h in fwd["history"]]
    rev_points = [base_point] + [(h["vote_share"], h["districts_won"]) for h in rev["history"]]

    merged = sorted(set(fwd_points + rev_points))
    return merged
#3. Updated plotting function to show both curves
def plot_step_and_complement(step_points, NUM_DISTRICTS, title="Seat–Vote Curve"):
    x, y = zip(*sorted(step_points))
    x, y = np.array(x), np.array(y)

    # Convert district counts to seat share (0–1)
    y = y / NUM_DISTRICTS

    plt.figure(figsize=(8, 6))
    
    # Original curve
    plt.step(x, y, where="post", color="black", label="Seat–Vote Curve", lw=2)

    # Complement curve: (1-x, 1-y)
    xc = 1 - x
    yc = 1 - y
    
    # Sort for proper plotting
    order = np.argsort(xc)
    plt.step(xc[order], yc[order], where="post",
             color="blue", linestyle="--", label="Complement Curve")

    plt.xlabel("Statewide Vote Share")
    plt.ylabel("Seat Share")
    plt.title(title)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.show()    

def compute_step_function_area(step_points, num_districts):
    """
    Computes the geometric area between the seat–vote curve (as seat share 0–1)
    and its complement curve.

    Parameters:
        step_points: list of (vote_share, districts_won) tuples
        num_districts: total number of districts

    Returns:
        Dictionary with:
            - area_between
            - area_step
            - area_complement
    """
    # Sort points by vote share
    points = sorted(step_points, key=lambda x: x[0])
    
    # Ensure endpoints at 0 and 1
    if points[0][0] > 0:
        points = [(0, 0)] + points
    if points[-1][0] < 1.0:
        points = points + [(1.0, points[-1][1])]
    
    # Convert to numpy arrays
    step_x, step_y = zip(*points)
    step_x = np.array(step_x)
    step_y = np.array(step_y) / num_districts   # Normalize to 0–1 seat share

    # Complement curve: g(x) = 1 - f(1-x)
    complement_x = 1 - step_x[::-1]
    complement_y = 1 - step_y[::-1]

    # Combine all x-breakpoints
    all_x = np.unique(np.concatenate([step_x, complement_x]))
    
    area_between = 0.0
    area_step = 0.0
    area_complement = 0.0

    def step_value_at(x, xs, ys):
        idx = np.searchsorted(xs, x, side='right') - 1
        idx = max(0, idx)
        return ys[idx]

    for i in range(len(all_x) - 1):
        x1 = all_x[i]
        x2 = all_x[i + 1]
        width = x2 - x1

        midpoint = (x1 + x2) / 2
        f_val = step_value_at(midpoint, step_x, step_y)
        g_val = step_value_at(midpoint, complement_x, complement_y)

        area_between += width * abs(f_val - g_val)
        area_step += width * f_val
        area_complement += width * g_val

    return {
        "area_between": area_between,
        "area_step": area_step,
        "area_complement": area_complement
    }
import itertools

def compute_pairwise_area(curve1, curve2, num_districts):
    """
    Computes the absolute area between the step curves of two different parties.
    """
    # 1. Normalize and Sort
    # curve format: [(vote_share, districts_won), ...]
    x1, y1 = zip(*sorted(curve1))
    x2, y2 = zip(*sorted(curve2))
    
    # Seat shares (0 to 1)
    y1 = np.array(y1) / num_districts
    y2 = np.array(y2) / num_districts

    # 2. Find every unique x-coordinate where either curve changes
    all_x = np.unique(np.concatenate([x1, x2]))
    if all_x[0] > 0: all_x = np.insert(all_x, 0, 0.0)
    if all_x[-1] < 1: all_x = np.append(all_x, 1.0)

    def get_val(x, xs, ys):
        idx = np.searchsorted(xs, x, side='right') - 1
        return ys[max(0, idx)]

    # 3. Integrate absolute difference
    total_area = 0.0
    for i in range(len(all_x) - 1):
        mid = (all_x[i] + all_x[i+1]) / 2
        width = all_x[i+1] - all_x[i]
        diff = abs(get_val(mid, x1, y1) - get_val(mid, x2, y2))
        total_area += width * diff
        
    return total_area
# ------------------------
# RUN SIMULATIONS
# ------------------------
all_efficiency_gaps = []
all_seat_counts = []
all_vote_shares = []       # Party 0 only, optional
all_seat_shares = []       # Party 0 only, optional
all_pr_vote_shares = []    # dict of all parties
all_pr_seat_shares = []    # dict of all parties
all_district_counts = []    # For potential further analysis
all_area_results = []    # For step function area analysis
all_step_curves = []       # Store step curves for potential plotting
plt.figure(figsize=(10,6))


'''    
for run in range(NUM_RUNS):
    district_labels = connected_districts(voters, NUM_DISTRICTS, VOTERS_PER_DISTRICT)
    district_counts = count_votes(district_labels)
'''
for run in range(NUM_RUNS):

    rng = np.random.default_rng()

    district_labels = connected_districts(
        voters,
        NUM_DISTRICTS,
        VOTERS_PER_DISTRICT,
        rng
    )

    district_counts = count_votes(district_labels)
# Efficiency gap
    gap = efficiency_gap(district_counts)
    all_efficiency_gaps.append(gap)

    # ------------------------
    # Proportional Representation (all parties)
    # ------------------------
    pr_vote_share = {party: 0 for party in PARTY_IDS}
    pr_seat_share = {party: 0 for party in PARTY_IDS}

    for d in district_counts:
        for party in PARTY_IDS:
            pr_vote_share[party] += d[party]

    # Normalize vote share
    total_votes = sum(pr_vote_share.values())
    for party in PARTY_IDS:
        pr_vote_share[party] /= total_votes

    # Seat share
    for d in district_counts:
        winner = max(d, key=d.get)
        pr_seat_share[winner] += 1
    for party in PARTY_IDS:
        pr_seat_share[party] /= NUM_DISTRICTS

    all_pr_vote_shares.append(pr_vote_share)
    all_pr_seat_shares.append(pr_seat_share)
    all_district_counts.append(district_counts)
    # -----------------------------------------
# STEP CURVES FOR EACH PARTY (per simulation)
# -----------------------------------------

    step_curves_this_run = {}
    area_results_this_run = {}

    for party in PARTY_IDS:
        curve = compute_step_curve_for_party(
            district_counts,
            party,
            PARTY_IDS,
            NUM_DISTRICTS
        )
        step_curves_this_run[party] = curve
        area_metrics = compute_step_function_area(curve, NUM_DISTRICTS)
        area_results_this_run[party] = area_metrics

# Store all curves for this simulation
    all_step_curves.append(step_curves_this_run)
    all_area_results.append(area_results_this_run)



# ------------------------
# SCATTER PLOT: Vote Share vs Seat Share
# ------------------------
plt.figure(figsize=(8,8))
for party in PARTY_IDS:
    vote_shares = [run[party] for run in all_pr_vote_shares]
    seat_shares = [run[party] for run in all_pr_seat_shares]
    plt.scatter(vote_shares, seat_shares, alpha=0.7, label=party)

plt.plot([0,1],[0,1], linestyle='--', color='black', label='Perfect Proportionality')
plt.xlabel("Vote Share")
plt.ylabel("Seat Share")
plt.title("Seat Share vs Vote Share per Party")
plt.xlim(0,1)
plt.ylim(0,1)
plt.gca().set_aspect('equal', adjustable='box')
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

# ------------------------
# SUMMARY
# ------------------------
print("\n----- REDISTRICTING SUMMARY -----")
print("Runs:", NUM_RUNS)
if NUM_PARTIES == 2:
    print("Mean Efficiency Gap:", np.nanmean(all_efficiency_gaps))
    print("Std Efficiency Gap:", np.nanstd(all_efficiency_gaps))
for party in PARTY_IDS:
    mean_vote = np.mean([run[party] for run in all_pr_vote_shares])
    mean_seat = np.mean([run[party] for run in all_pr_seat_shares])
    print(f"{party}: Mean Vote Share = {mean_vote:.3f}, Mean Seat Share = {mean_seat:.3f}")

# ------------------------------------------------------------
# GENERATE DETAILED DISTRICT-BY-DISTRICT WINNER TABLE
# ------------------------------------------------------------
detailed_results = []

for run_idx in range(NUM_RUNS):
    # Get the district counts for this specific simulation
    districts_in_run = all_district_counts[run_idx]
    
    for dist_idx, counts in enumerate(districts_in_run):
        # Identify the winner of this specific district
        winner = max(counts, key=counts.get)
        winner_votes = counts[winner]
        total_dist_votes = sum(counts.values())
        win_margin = winner_votes / total_dist_votes
        
        detailed_results.append({
            "Simulation": run_idx + 1,
            "District": dist_idx + 1,
            "Winner": winner,
            "Winner Votes": winner_votes,
            "Total Votes": total_dist_votes,
            "Win Margin %": f"{win_margin * 100:.1f}%"
        })

# Create the DataFrame
df_detailed = pd.DataFrame(detailed_results)

# Display options to ensure we can see the data clearly
pd.set_option('display.max_rows', 20) # Shows first and last 10 rows
print("\n--- DISTRICT-LEVEL WINNERS PER SIMULATION ---")
print(df_detailed.to_string(index=False))

# ------------------------------------------------------------
# OPTIONAL: Summary Table of "Total Wins" per Party
# ------------------------------------------------------------
print("\n--- TOTAL DISTRICTS WON ACROSS ALL RUNS ---")
print(df_detailed['Winner'].value_counts())
plot_step_and_complement(all_step_curves[9]["Party 0"], NUM_DISTRICTS)
# Access the dictionary for the first run, then the specific party
party_0_run_0_metrics = all_area_results[0]["Party 1"]

# Now you can print individual components
print("Area Step:", party_0_run_0_metrics["area_step"])
print("Area Complement:", party_0_run_0_metrics["area_complement"])
print("Area Difference (Bias):", party_0_run_0_metrics["area_between"])


print(all_efficiency_gaps)
all_efficiency_gaps[2][4]
# ------------------------------------------------------------
# RUN PAIRWISE SIMULATIONS
# ------------------------------------------------------------
pairwise_results = []
party_pairs = list(itertools.combinations(PARTY_IDS, 2))

for run_idx in range(NUM_RUNS):
    # (Assuming district_counts and curves are generated here as in your main code)
    # Get all party curves for this run
    current_curves = all_step_curves[run_idx] 
    
    for p1, p2 in party_pairs:
        area_gap = compute_pairwise_area(current_curves[p1], current_curves[p2], NUM_DISTRICTS)
        
        pairwise_results.append({
            "Run": run_idx + 1,
            "Comparison": f"{p1} vs {p2}",
            "Area Gap": area_gap
        })

df_pairs = pd.DataFrame(pairwise_results)
print("\n--- PAIRWISE AREA DIFFERENCES ---")
print(df_pairs.to_string(index=False))
def plot_pairwise_comparison(run_idx, party_a, party_b):
    curve_a = all_step_curves[run_idx][party_a]
    curve_b = all_step_curves[run_idx][party_b]
    
    xa, ya = zip(*curve_a)
    xb, yb = zip(*curve_b)
    
    plt.figure(figsize=(8, 5))
    plt.step(xa, np.array(ya)/NUM_DISTRICTS, where='post', label=party_a, color=PARTY_COLORS[party_a])
    plt.step(xb, np.array(yb)/NUM_DISTRICTS, where='post', label=party_b, color=PARTY_COLORS[party_b])
    
    plt.fill_between(xa, np.array(ya)/NUM_DISTRICTS, step="post", alpha=0.1) # Optional shading
    plt.title(f"Pairwise Comparison: {party_a} vs {party_b} (Run {run_idx+1})")
    plt.xlabel("Vote Share")
    plt.ylabel("Seat Share")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Example: Plotting first run for the first pair
plot_pairwise_comparison(0, party_pairs[0][0], party_pairs[0][1])
def plot_party_overlays(all_step_curves, PARTY_IDS, num_runs_to_plot=5):
    """
    Creates one figure per party, overlaying multiple simulation runs 
    of both the Seat-Vote and Complement curves.
    """
    num_districts = NUM_DISTRICTS # Uses global NUM_DISTRICTS
    
    for party in PARTY_IDS:
        plt.figure(figsize=(10, 7))
        
        # Determine how many runs we can actually plot
        runs_available = min(num_runs_to_plot, len(all_step_curves))
        
        for run_idx in range(runs_available):
            # 1. Extract data for this party in this run
            curve_points = all_step_curves[run_idx][party]
            x, y = zip(*sorted(curve_points))
            x = np.array(x)
            y = np.array(y) / num_districts  # Normalize to Seat Share (0-1)
            
            # 2. Calculate Complement coordinates: (1-x, 1-y)
            xc = 1 - x
            yc = 1 - y
            # Sort complement for step plotting
            c_order = np.argsort(xc)
            xc_sorted = xc[c_order]
            yc_sorted = yc[c_order]
            
            # 3. Plotting
            # We only want one label in the legend per group, not for every line
            label_sv = f"Seat-Vote (Runs 1-{runs_available})" if run_idx == 0 else None
            label_comp = f"Complement (Runs 1-{runs_available})" if run_idx == 0 else None
            
            # Plot Seat-Vote Curve (Solid)
            plt.step(x, y, where="post", color="black", alpha=0.3, lw=1.5, label=label_sv)
            
            # Plot Complement Curve (Dashed)
            plt.step(xc_sorted, yc_sorted, where="post", color="blue", 
                     linestyle="--", alpha=0.3, lw=1.5, label=label_comp)

        # Reference: Perfect Symmetry line
        plt.plot([0, 1], [0, 1], color="red", linestyle=":", alpha=0.6, label="Ideal Symmetry")

        # Formatting
        plt.title(f"Overlay Analysis: {party} ({runs_available} Simulations)", fontsize=15)
        plt.xlabel("Statewide Vote Share", fontsize=12)
        plt.ylabel("Seat Share", fontsize=12)
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        plt.grid(True, linestyle="--", alpha=0.4)
        plt.legend(loc="upper left")
        
        plt.tight_layout()
        plt.show()

# ------------------------
# EXECUTION
# ------------------------
plot_party_overlays(all_step_curves, PARTY_IDS, num_runs_to_plot=10)
#lets update the step curve so that the curves are compared between parties not x and 1-x and y and 1-y. 

#Efficiency Gap Histograms per District

# Number of districts
n_districts = NUM_DISTRICTS

# Grid layout
n_cols = 2
n_rows = math.ceil(n_districts / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4*n_rows))
axes = axes.flatten()

# Plot histograms
for d in range(n_districts):
    district_gaps = [all_efficiency_gaps[run][d] for run in range(NUM_RUNS)]
    
    axes[d].hist(district_gaps,bins=NUM_RUNS,edgecolor='black',alpha=0.7,density=True ) # <-- normalize
    axes[d].set_title(f"District {d}")
    axes[d].set_xlabel("Efficiency Gap")
    axes[d].set_ylabel("Density")
    #axes[d].set_ylabel("Frequency")
    #axes[d].set_ylim(0, NUM_RUNS)  # FIXED vertical axis
    axes[d].grid(True, linestyle="--", alpha=0.4)

# Remove empty subplots
for j in range(d+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()
import pandas as pd

# all_efficiency_gaps is currently [run][district]

# Convert to DataFrame: rows=districts, columns=runs
df_efficiency = pd.DataFrame(all_efficiency_gaps).T

# Rename columns for clarity
df_efficiency.columns = [f"Run {i}" for i in range(NUM_RUNS)]
df_efficiency.index = [f"District {i}" for i in range(NUM_DISTRICTS)]

# Display
print(df_efficiency)


# 1. First, collect all data and find the global maximum frequency
all_data_points = {}
max_freq = 0

for party in PARTY_IDS:
    # Extract area_between for this party
    data = [run[party]["area_between"] for run in all_area_results if party in run]
    all_data_points[party] = data
    
    if data:
        # Calculate a temporary histogram to find the max height for this party
        counts, _ = np.histogram(data, bins=10) # Using 10 bins as a standard for comparison
        max_freq = max(max_freq, np.max(counts))

# 2. Setup Grid Layout
n_parties = len(PARTY_IDS)
n_cols = 2
n_rows = math.ceil(n_parties / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows), sharex=True)
axes = axes.flatten() if n_parties > 1 else [axes]

# 3. Plot with synchronized Y-axis
for i, party in enumerate(PARTY_IDS):
    data = all_data_points[party]
    
    if data:
        # Plot histogram
        axes[i].hist(data, bins=10, alpha=0.7, color='skyblue', edgecolor='black', density=True) # <-- density=True for normalized histogram
        
        # Calculate mean for title
        mean_val = np.mean(data)
        axes[i].set_title(f"{party} (Mean: {mean_val:.4f})", fontsize=13)
        
        # APPLY CONSISTENT Y-LIMIT
        # Adding 1 to max_freq gives a little breathing room at the top
        #axes[i].set_ylim(0, max_freq + 1)
        
        axes[i].set_xlabel("Area Difference (|f - g|)")
        axes[i].set_ylabel("Density")
        axes[i].grid(True, linestyle="--", alpha=0.4)

# Hide extra subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()
# ------------------------------------------------------------
# GENERATE RUN-BY-RUN AREA DIFFERENCE TABLE WITH STATS
# ------------------------------------------------------------

# 1. First, build the raw data dictionary (keeping values as floats for math)
table_data = {"Party": PARTY_IDS}

# Create a list to help calculate row-wise stats later
all_values_matrix = []

for party in PARTY_IDS:
    party_row = []
    for run_idx in range(NUM_RUNS):
        area_val = all_area_results[run_idx][party]["area_between"]
        party_row.append(area_val)
    all_values_matrix.append(party_row)

# 2. Build the DataFrame
df_runs = pd.DataFrame(all_values_matrix, columns=[f"Run {i+1}" for i in range(NUM_RUNS)])

# 3. Add the Mean and Std Dev columns
df_runs["Mean"] = df_runs.mean(axis=1)
df_runs["Std Dev"] = df_runs.std(axis=1)

# 4. Insert the Party names at the start
df_runs.insert(0, "Party", PARTY_IDS)

# 5. Format for printing (round to 4 decimal places)
print("\n--- AREA DIFFERENCE PER SIMULATION (WITH STATS) ---")
print(df_runs.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# 6. Transposed version (often easier to read with many runs)
# print("\n--- TRANSPOSED VIEW ---")
# print(df_runs.set_index("Party").T)
# ------------------------
# VISUALIZE ONE RUN (NO OVERLAP)
# ------------------------

district_labels = connected_districts(voters, NUM_DISTRICTS, VOTERS_PER_DISTRICT, rng)

# Build KD-tree for fast nearest lookup
tree = cKDTree(voters)

# Create grid of every pixel
xx, yy = np.meshgrid(np.arange(WIDTH), np.arange(HEIGHT))
grid_points = np.column_stack((xx.ravel(), yy.ravel()))

# Assign each pixel to nearest voter
_, nearest_voter = tree.query(grid_points)
pixel_districts = district_labels[nearest_voter]
pixel_map = pixel_districts.reshape((HEIGHT, WIDTH))

plt.figure(figsize=(10,7))

# Lightly color districts
plt.imshow(pixel_map, origin='lower', cmap='tab20', alpha=0.35)

# Draw district boundaries
plt.contour(
    pixel_map,
    levels=np.arange(NUM_DISTRICTS)+0.5,
    colors='black',
    linewidths=1,
    origin='lower'
)

# Plot voters on top
for i, party in enumerate(PARTY_IDS):
    mask = voter_parties == i
    plt.scatter(
        voters[mask,0],
        voters[mask,1],
        color=PARTY_COLORS[party],
        label=party,
        s=20
    )

plt.title("Voters and Districts (True Boundary Lines)")
plt.xlabel("X Coordinate")
plt.ylabel("Y Coordinate")
plt.legend()
plt.show()




## Wasted Vote Ratio Table

In addition to the efficiency gap, we compute a **wasted vote ratio** for each party in every simulation.

For party $p$:


$\text{Wasted Vote Ratio}_p =
\frac{\text{Total Wasted Votes}_p}{\text{Total Votes}}$


where

$
\text{Total Wasted Votes}_p =
\sum_{d=1}^{D} W_{p,d}
$

and

- $D$ = number of districts  
- $W_{p,d}$ = wasted votes for party $p$ in district $d$

After computing wasted votes for each district, we sum them across all districts.
This gives one total wasted votes number per party for the simulation run.

This produces a table with the following structure:

| Run | Party 0 | Party 1 | Party 2 |
|----|----|----|----|
| 1 | 0.13 | 0.29 | 0.25 |
| 2 | ... | ... | ... |

Each row represents a **different redistricting simulation**, while each column shows the **fraction of total votes that were wasted for that party**.

Higher wasted vote ratios indicate that a party's supporters are **either heavily concentrated in a few districts (packing)** or **spread thinly across many districts (cracking)**.

This table allows us to analyze how different district maps affect the **distribution of wasted votes across parties**.
# ------------------------------------------------------------
# WASTED VOTE RATIOS PER PARTY PER RUN
# ------------------------------------------------------------

wasted_vote_table = []

for run_idx in range(NUM_RUNS):

    district_counts = all_district_counts[run_idx]

    # Track wasted votes per party
    wasted_votes = {party: 0 for party in PARTY_IDS}
    total_votes = 0

    for district in district_counts:

        district_total = sum(district.values())
        total_votes += district_total

        # Winner of district
        winner = max(district, key=district.get)

        for party in PARTY_IDS:

            votes = district[party]

            if party == winner:
                wasted = votes - (district_total / NUM_PARTIES)
            else:
                wasted = votes

            wasted_votes[party] += wasted

    # Convert to ratios
    ratios = {party: wasted_votes[party] / total_votes for party in PARTY_IDS}

    row = {"Run": run_idx + 1}
    row.update(ratios)

    wasted_vote_table.append(row)

df_wasted = pd.DataFrame(wasted_vote_table)

print("\n--- WASTED VOTE RATIO (WASTED / TOTAL VOTES) ---")
print(df_wasted.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n--- MEAN WASTED VOTE RATIO ACROSS RUNS ---")
print(df_wasted.drop(columns="Run").mean())



# ------------------------------------------------------------
# EXPECTED WASTED VOTE RATIOS (ANALYTIC)
# ------------------------------------------------------------

# Compute total votes per party across all districts
total_votes_per_party = {party: 0 for party in PARTY_IDS}
total_votes_all = 0

# First, compute overall vote fractions
for district in all_district_counts[0]:  # use first run as representative
    district_total = sum(district.values())
    total_votes_all += district_total
    for party in PARTY_IDS:
        total_votes_per_party[party] += district[party]

# Compute vote shares
vote_shares = {party: total_votes_per_party[party] / total_votes_all for party in PARTY_IDS}

# Expected wasted vote ratios
expected_wasted = {}
for party in PARTY_IDS:
    p = vote_shares[party]
    expected_wasted[party] = (p * NUM_PARTIES - 1) / NUM_PARTIES  # approximate per-district formula

print("\n--- EXPECTED WASTED VOTE RATIO (ANALYTIC) ---")
for party, val in expected_wasted.items():
    print(f"Party {party}: {val:.4f}")
## Efficiency Gap

The **Efficiency Gap (EG)** is a common metric used to detect partisan bias in districting plans.  
It measures the difference in **wasted votes** between political parties.

A **wasted vote** is defined as:

- **For the winning party:**  
  votes beyond the threshold needed to win.
- **For losing parties:**  
  all votes cast for that party.

For a district with total votes \(V\) and \(n\) parties:

- Winning party wasted votes:
  
$
W_{\text{winner}} = v_{\text{winner}} - \frac{V}{n}
$

- Losing party wasted votes:

$
W_{\text{loser}} = v_{\text{loser}}
$

The **Efficiency Gap** compares wasted votes between parties:

$
EG = \frac{W_A - W_B}{\text{Total Votes}}
$

where

- \(W_A\) = total wasted votes for Party A  
- \(W_B\) = total wasted votes for Party B  

Interpretation:

- **Positive EG:** Party A is advantaged
- **Negative EG:** Party B is advantaged
- **EG ≈ 0:** the districting plan is approximately symmetric

In this project the efficiency gap is computed **per district for each redistricting simulation**.

---

# ------------------------------------------------------------
# MAJORITY PARTY EFFICIENCY GAP PER RUN
# ------------------------------------------------------------

gap_rows = []

for run_idx in range(NUM_RUNS):

    district_counts = all_district_counts[run_idx]

    wasted_votes = {party: 0 for party in PARTY_IDS}
    total_votes = 0

    for district in district_counts:

        district_total = sum(district.values())
        total_votes += district_total

        winner = max(district, key=district.get)

        for party in PARTY_IDS:

            votes = district[party]

            if party == winner:
                wasted = votes - (district_total / NUM_PARTIES)
            else:
                wasted = votes

            wasted_votes[party] += wasted

    # Majority party
    majority_party = max(wasted_votes, key=wasted_votes.get)

    wasted_majority = wasted_votes[majority_party]
    wasted_others = sum(v for p, v in wasted_votes.items() if p != majority_party)

    gap_value = (wasted_majority - wasted_others) / total_votes

    gap_rows.append({
        "Run": run_idx + 1,
        "Majority Party": majority_party,
        "Gap": gap_value
    })

df_gap = pd.DataFrame(gap_rows)

print("\n--- MAJORITY PARTY EFFICIENCY GAP ---")
print(df_gap.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(np.bincount(district_labels))

SyntaxError: invalid syntax (3334489133.py, line 892)